# NB01 – Daten Laden
### CAS Information Engineering – Scripting Project
**Gruppe:** SC26_Gruppe_2 | **Verantwortlich:** Person A | **Datum:** März 2026

---
Lädt echte Rohdaten von externen Quellen und speichert sie unverändert in `data/raw/`.  
Fehler → `data/missing.txt` + RuntimeError mit Hinweis auf NB01b (Simulation).

Alle Parameter werden aus `config.json` gelesen — nie direkt im Code anpassen.

### Ausgabe → `data/raw/`
| Datei | Spalten | Quelle |
|-------|---------|--------|
| `ch_spot_prices_raw.csv` | `timestamp` (UTC ISO8601), `price_eur_mwh` | ENTSO-E DS1 |
| `ch_netzlast_raw.csv` | `timestamp` (UTC ISO8601), `load_gw` | ENTSO-E DS2 |

---
| [↑ Projektübersicht](00_Project_Overview.ipynb) | [→ NB02 Daten Analyse](02_Daten_Analyse.ipynb) |
|:---|---:|


In [1]:
# ── Check, if used Libraries need to be installed ───────────────────────────
import subprocess, sys
for imp, pkg in [('pandas','pandas'),('requests','requests'),('numpy','numpy'),
                 ('entsoe','entsoe-py')]:
    try: __import__(imp)
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])


In [2]:
# ── Projektstruktur & Konfiguration ─────────────────────────────────────────
import os, sys, warnings, datetime, json as _json
import numpy  as np
import pandas as pd
import requests
warnings.filterwarnings('ignore')

# ── config.json laden (Single Source of Truth) ───────────────────────────────
# Schalter NIE direkt hier setzen — immer in config.json anpassen.
_cfg_path = 'config.json'
if not os.path.exists(_cfg_path):
    raise FileNotFoundError(
        'config.json nicht gefunden. '
        'Bitte ins Projektverzeichnis legen (siehe NB00b Sektion 5).')
with open(_cfg_path) as _f:
    CFG = _json.load(_f)

# ── Aliases (nur lesend — Quelle bleibt config.json) ─────────────────────────
MODE         = CFG['mode']            # 'data' | 'sim'
FORCE_RELOAD = CFG['force_reload']    # dict mit allen Keys

# ── Datenzeitraum ────────────────────────────────────────────────────────────
_start   = CFG['daten']['start_year']
_end     = CFG['daten']['end_year']
START_YEAR = int(_start)
END_YEAR   = datetime.datetime.now().year if str(_end) == 'heute' else int(_end)
YEARS      = list(range(START_YEAR, END_YEAR + 1))
print(f'config.json geladen | MODE={MODE}')
print(f'Datenzeitraum: {START_YEAR}–{END_YEAR} | Jahre: {YEARS}')

def needs_download(path, min_kb, key):
    """True wenn Datei fehlt, zu klein, oder FORCE_RELOAD gesetzt."""
    if FORCE_RELOAD.get(key, False): return True
    if not os.path.exists(path): return True
    return os.path.getsize(path) / 1024 < min_kb

# ── Verzeichnisstruktur ───────────────────────────────────────────────────────
DIR_RAW          = os.path.join(MODE, 'raw')
DIR_PROCESSED    = os.path.join(MODE, 'processed')
DIR_INTERMEDIATE = os.path.join(MODE, 'intermediate')
DATAINDEX        = 'dataindex.csv'

for d in [DIR_RAW, DIR_PROCESSED, DIR_INTERMEDIATE,
          os.path.join('data','raw'), os.path.join('data','processed'),
          os.path.join('data','intermediate'),
          os.path.join('data','intermediate','pessimistisch'),
          os.path.join('data','intermediate','realistisch'),
          os.path.join('data','intermediate','optimistisch'),
          os.path.join('sim','raw'),  os.path.join('sim','processed'),
          os.path.join('sim','intermediate'),
          os.path.join('output','charts'),
          os.path.join('output','kuer','pessimistisch'),
          os.path.join('output','kuer','realistisch'),
          os.path.join('output','kuer','optimistisch')]:
    os.makedirs(d, exist_ok=True)

print(f'raw          : {os.path.abspath(DIR_RAW)}')
print(f'processed    : {os.path.abspath(DIR_PROCESSED)}')
print(f'intermediate : {os.path.abspath(DIR_INTERMEDIATE)}')


config.json geladen | MODE=data
Datenzeitraum: 2023–2026 | Jahre: [2023, 2024, 2025, 2026]
raw          : C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage\data\raw
processed    : C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage\data\processed
intermediate : C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage\data\intermediate


In [3]:
# ── dataindex.csv Helper ───────────────────────────────────────────────────────
def log_dataindex(filename, source_url, local_path, data_type,
                  rows=None, size_kb=None, status='active', note=''):
    """
    Zentrales Datenquellen-Register. Jeder Eintrag bleibt als Historie erhalten.
    status: active | superseded | error | partial
    """
    ts = datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z'
    if os.path.exists(DATAINDEX):
        df_idx = pd.read_csv(DATAINDEX)
        mask   = (df_idx['filename'] == filename) & (df_idx['status'] == 'active')
        if mask.any():
            df_idx.loc[mask, 'status']        = 'superseded'
            df_idx.loc[mask, 'superseded_at'] = ts
    else:
        df_idx = pd.DataFrame(columns=['timestamp','filename','source_url','local_path',
                                        'data_type','rows','size_kb','status','superseded_at','note'])
    row = {'timestamp': ts, 'filename': filename, 'source_url': source_url,
           'local_path': local_path, 'data_type': data_type, 'rows': rows,
           'size_kb': round(size_kb,1) if size_kb else None,
           'status': status, 'superseded_at': '', 'note': note}
    pd.concat([df_idx, pd.DataFrame([row])], ignore_index=True).to_csv(DATAINDEX, index=False)
    print(f'  dataindex: {filename} [{status}]')

def log_missing(source, reason):
    ts = datetime.datetime.utcnow().isoformat(timespec='seconds')
    with open(os.path.join('data','missing.txt'), 'a') as f:
        f.write(f'[{ts}] MISSING {source}: {reason}\n')
    log_dataindex(os.path.basename(source), source, '', 'raw', status='error', note=reason)
    print(f'  missing.txt: {reason}')

print('dataindex Helper bereit.')


dataindex Helper bereit.


---
## Datensatz 1: ENTSO-E Day-Ahead Spot-Preise CH

**Quelle:** ENTSO-E Transparency Platform — `transparency.entsoe.eu`  
**Methode:** REST-API via `entsoe-py` Python-Bibliothek (`query_day_ahead_prices`)  
**Zone:** `10YCH-SWISSGRIDZ` (Schweizer Bietungszone)  
**Format:** Stündliche Preise [EUR/MWh], UTC-Index  
**Zweck:** Hauptdatensatz für Dispatch-Simulation und Wirtschaftlichkeitsrechnung.  
Der Intra-Tag-Spread (p75 − p25) ist der direkte Erlöstreiber der Grid-Arbitrage.


In [4]:
# API-Key aus config.json (api_keys.entsoe) — nie im Code hardcoden
ENTSOE_API_KEY = (CFG.get('api_keys', {}).get('entsoe', '')
                  or os.environ.get('ENTSOE_API_KEY', ''))

PRICES_FILE = os.path.join(DIR_RAW, 'ch_spot_prices_raw.csv')

# ── Connectivity-Check (web-api Endpunkt, nicht nur Portal-Startseite) ───────
# Hinweis: transparency.entsoe.eu (HTTP 200) ≠ web-api.tp.entsoe.eu (kann 503 sein)
# Daher direkt den API-Endpunkt pingen statt die Webseite.
try:
    r = requests.get(
        'https://web-api.tp.entsoe.eu/api',
        params={'securityToken': ENTSOE_API_KEY, 'documentType': 'A44'},
        timeout=10)
    # 400 = Endpunkt erreichbar (Pflichtparameter fehlen — erwartetes Verhalten)
    # 401 = Endpunkt erreichbar, Key ungültig
    # 503 = Server temporär überlastet — ERREICHBAR, Retry übernimmt
    # → portal_ok = True für alle Codes ausser Netzwerkfehler
    portal_ok = r.status_code != 0
    print(f'ENTSO-E API-Endpunkt: HTTP {r.status_code} | '
          f'Key vorhanden: {bool(ENTSOE_API_KEY)}')
    if r.status_code == 503:
        print('  → 503: Server überlastet, aber erreichbar — Retry-Logik übernimmt.')
    elif r.status_code == 401:
        print('  → 401: API-Key ungültig — bitte prüfen.')
        portal_ok = False
except Exception as e:
    portal_ok = False
    print(f'ENTSO-E nicht erreichbar (Netzwerk?): {e}')

# ── Hilfsfunktion: Download eines Jahres mit 503-Retry ───────────────────────
def _fetch_prices_year(client, year, max_retries=3, wait_s=20):
    """Lädt ein Jahr Day-Ahead-Preise mit Retry bei 503."""
    import time
    from requests.exceptions import HTTPError
    start = pd.Timestamp(f'{year}-01-01', tz='Europe/Zurich')
    end   = pd.Timestamp(f'{year}-12-31 23:00', tz='Europe/Zurich')
    for attempt in range(1, max_retries + 1):
        try:
            ts = client.query_day_ahead_prices('CH', start=start, end=end)
            return ts
        except HTTPError as e:
            if '503' in str(e) and attempt < max_retries:
                print(f'  Jahr {year}: 503 → Versuch {attempt}/{max_retries}, '
                      f'warte {wait_s}s...')
                time.sleep(wait_s)
            else:
                raise  # Anderer Fehler oder max Retries erreicht

if not needs_download(PRICES_FILE, 10, 'spot_prices'):
    print(f'Preisdaten vorhanden ({os.path.getsize(PRICES_FILE)/1024:.0f} KB) – kein Re-Download.')
    df_prices = pd.read_csv(PRICES_FILE)

elif ENTSOE_API_KEY and portal_ok:
    print(f'Lade ENTSO-E CH Preise {START_YEAR}–{END_YEAR} ({len(YEARS)} Jahre, max. 3 Retries bei 503)...')
    from entsoe import EntsoePandasClient
    client = EntsoePandasClient(api_key=ENTSOE_API_KEY)
    frames = []
    for year in YEARS:
        print(f'  Lade {year}...', end=' ')
        ts_year = _fetch_prices_year(client, year)
        frames.append(ts_year)
        print(f'{len(ts_year):,} h OK')
    ts = pd.concat(frames).sort_index()
    ts = ts[~ts.index.duplicated(keep='first')]  # Überlapp-Schutz
    df_prices = ts.rename('price_eur_mwh').to_frame().reset_index()
    df_prices.columns = ['timestamp', 'price_eur_mwh']
    df_prices['timestamp'] = pd.to_datetime(df_prices['timestamp'], utc=True)
    df_prices.to_csv(PRICES_FILE, index=False, encoding='utf-8')
    kb = os.path.getsize(PRICES_FILE) / 1024
    log_dataindex('ch_spot_prices_raw.csv',
                  'https://transparency.entsoe.eu', PRICES_FILE, 'raw',
                  rows=len(df_prices), size_kb=kb, note=f'ENTSO-E API, entsoe-py, {START_YEAR}-{END_YEAR}')
    print(f'Gespeichert: {PRICES_FILE} | {len(df_prices):,} h | {kb:.0f} KB')
else:
    reason = 'Kein API-Key' if not ENTSOE_API_KEY else 'Portal/API nicht erreichbar (503?)'
    log_missing('ch_spot_prices_raw.csv', reason)
    raise RuntimeError(
        f'{reason}\n'
        '→ Bei 503: In 15-30 Min erneut versuchen (ENTSO-E hat gelegentlich Wartungsfenster).\n'
        '→ API-Key beantragen: transparency@entsoe.eu, Betreff: Restful API access\n'
        '→ Oder sofort weiterarbeiten: NB1b_Daten_Sim ausführen.')

print(f'Qualitaet: {len(df_prices):,} Zeilen | '
      f'Min: {df_prices["price_eur_mwh"].min():.1f} / '
      f'Max: {df_prices["price_eur_mwh"].max():.1f} EUR/MWh')


ENTSO-E nicht erreichbar (Netzwerk?): HTTPSConnectionPool(host='web-api.tp.entsoe.eu', port=443): Read timed out. (read timeout=10)
Preisdaten vorhanden (925 KB) – kein Re-Download.
Qualitaet: 28,437 Zeilen | Min: -427.5 / Max: 318.0 EUR/MWh


In [5]:
# ── Verifikation DS1: Spot-Preise ───────────────────────────────────────────
print(f'Shape   : {df_prices.shape}')
print(f'Zeitraum: {df_prices["timestamp"].min()} – {df_prices["timestamp"].max()}')
print(f'Nulls   : {df_prices.isnull().sum().sum()}')
print(f'Range   : {df_prices["price_eur_mwh"].min():.1f} – '
      f'{df_prices["price_eur_mwh"].max():.1f} EUR/MWh')
df_prices.head(3)


Shape   : (28437, 2)
Zeitraum: 2022-12-31 23:00:00+00:00 – 2026-03-31 21:00:00+00:00
Nulls   : 0
Range   : -427.5 – 318.0 EUR/MWh


,timestamp,price_eur_mwh
0,2022-12-31 23:00:00+00:00,0.03
1,2023-01-01 00:00:00+00:00,-7.25
2,2023-01-01 01:00:00+00:00,-3.99


---
## Datensatz 2: CH Netzlast (Systemlast)

**Quelle:** ENTSO-E Transparency Platform — `transparency.entsoe.eu`  
**Methode:** REST-API via `entsoe-py` (`query_load('CH')`)  
**Identisch mit DS1:** Gleicher API-Key und Endpunkt — eine Moodle-URL für DS1+DS2.  
**Format:** Stündliche Systemlast [MW→GW], UTC-Index  
**Zweck:** Netzentlastungsmodell — aggregierte Batterien sollen die Spitzenlast senken.  
Koinzidenz mit Preishochpunkten bestätigt Business Case (Import teuer = Netz voll).


In [6]:
# ── Datensatz 2: CH Netzlast via ENTSO-E ────────────────────────────────────
# Gleicher API-Key wie für Preise.
# query_load('CH') liefert die stündliche Systemlast des Schweizer Regelblocks [MW].
# Jahresweiser Download mit Retry bei 503 — identisch zur DS1-Logik.

LOAD_FILE = os.path.join(DIR_RAW, 'ch_netzlast_raw.csv')

def _fetch_load_year(client, year, max_retries=3, wait_s=20):
    """Lädt ein Jahr Netzlast mit Retry bei 503 Service Unavailable.
    Identische Strategie wie _fetch_prices_year() für DS1.
    ENTSO-E gibt bei Überlastung 503 zurück — jahresweiser Abruf mit Wartezeit
    umgeht das Problem zuverlässiger als ein einziger Gesamtrequest.
    """
    import time
    from requests.exceptions import HTTPError
    start = pd.Timestamp(f'{year}-01-01', tz='Europe/Zurich')
    end   = pd.Timestamp(f'{year}-12-31 23:00', tz='Europe/Zurich')
    for attempt in range(1, max_retries + 1):
        try:
            ts = client.query_load('CH', start=start, end=end)
            return ts
        except HTTPError as e:
            if '503' in str(e) and attempt < max_retries:
                print(f'  Jahr {year}: 503 → Versuch {attempt}/{max_retries}, '
                      f'warte {wait_s}s...')
                time.sleep(wait_s)
            else:
                raise

if not needs_download(LOAD_FILE, 10, 'netzlast'):
    print(f'Lastdaten vorhanden ({os.path.getsize(LOAD_FILE)/1024:.0f} KB) – kein Re-Download.')
    df_load = pd.read_csv(LOAD_FILE)

elif ENTSOE_API_KEY and portal_ok:
    print(f'Lade CH Netzlast {START_YEAR}–{END_YEAR} ({len(YEARS)} Jahre, jahresweise, max. 3 Retries bei 503)...')
    try:
        from entsoe import EntsoePandasClient
        client = EntsoePandasClient(api_key=ENTSOE_API_KEY)
        frames = []
        for year in YEARS:
            print(f'  Lade {year}...', end=' ')
            ts_year = _fetch_load_year(client, year)

            # query_load gibt je nach entsoe-py Version DataFrame oder Series zurück
            if isinstance(ts_year, pd.DataFrame):
                load_col = ts_year.columns[0]
                df_year  = ts_year[[load_col]].reset_index()
                df_year.columns = ['timestamp', 'load_mw']
            else:
                df_year = ts_year.to_frame(name='load_mw').reset_index()
                df_year.columns = ['timestamp', 'load_mw']

            frames.append(df_year)
            print(f'{len(df_year):,} h OK')

        df_load = pd.concat(frames, ignore_index=True)
        df_load = df_load[~df_load['timestamp'].duplicated(keep='first')]  # Überlapp-Schutz
        df_load['timestamp'] = pd.to_datetime(df_load['timestamp'], utc=True)
        df_load['load_gw']   = (df_load['load_mw'] / 1000).round(4)
        df_load = df_load[['timestamp', 'load_gw']].sort_values('timestamp').reset_index(drop=True)

        df_load.to_csv(LOAD_FILE, index=False, encoding='utf-8')
        kb = os.path.getsize(LOAD_FILE) / 1024
        log_dataindex(
            'ch_netzlast_raw.csv',
            'https://transparency.entsoe.eu (query_load CH)',
            LOAD_FILE, 'raw',
            rows=len(df_load), size_kb=kb,
            note=f'ENTSO-E query_load, jahresweise, MW→GW, {START_YEAR}–{END_YEAR}')
        print(f'Gespeichert: {LOAD_FILE} | {len(df_load):,} h | {kb:.0f} KB')

    except Exception as e:
        log_missing('ch_netzlast_raw.csv', str(e))
        raise RuntimeError(
            f'ENTSO-E query_load fehlgeschlagen: {e}\n'
            '→ NB01b_Daten_Sim ausführen (MODE=\'sim\' in config.json setzen).')
else:
    reason = 'Kein API-Key' if not ENTSOE_API_KEY else 'Portal nicht erreichbar'
    log_missing('ch_netzlast_raw.csv', reason)
    raise RuntimeError(f'{reason} → NB01b_Daten_Sim ausführen.')

# Qualitätsprüfung: CH Systemlast typisch 5–12 GW
df_load_check = pd.read_csv(LOAD_FILE)
assert len(df_load_check) > 8000, f'Zu wenig Zeilen: {len(df_load_check)}'
assert df_load_check['load_gw'].between(2, 20).mean() > 0.98, 'Lastdaten ausserhalb plausiblem Bereich'
print(f'Qualität OK | Median: {df_load_check["load_gw"].median():.2f} GW | '
      f'{len(df_load_check):,} Zeilen')


Lastdaten vorhanden (941 KB) – kein Re-Download.
Qualität OK | Median: 7.07 GW | 28,429 Zeilen


In [7]:
# ── Verifikation DS2: Netzlast ──────────────────────────────────────────────
print(f'Shape   : {df_load.shape}')
print(f'Zeitraum: {df_load["timestamp"].min()} – {df_load["timestamp"].max()}')
print(f'Nulls   : {df_load.isnull().sum().sum()}')
print(f'Range   : {df_load["load_gw"].min():.2f} – {df_load["load_gw"].max():.2f} GW')
df_load.head(3)


Shape   : (28429, 2)
Zeitraum: 2022-12-31 23:00:00+00:00 – 2026-03-30 16:00:00+00:00
Nulls   : 0
Range   : 2.37 – 15.87 GW


,timestamp,load_gw
0,2022-12-31 23:00:00+00:00,7.6668
1,2023-01-01 00:00:00+00:00,7.3730
2,2023-01-01 01:00:00+00:00,7.4643


---
## Abschluss & Verifikation


In [8]:
# -- Transfer: Datenzeitraum in transfer.json schreiben -----------------------
# Wird von NB02–NB15 gelesen; muss nach dem Laden der Preisdaten stehen,
# damit n_years aus echten Daten stammt.
import os as _os, json as _json
_tf_path = 'transfer.json'
_tf = _json.loads(open(_tf_path).read() or '{}') if _os.path.exists(_tf_path) and _os.path.getsize(_tf_path) > 0 else {}
_tf['datenzeitraum'] = {
    'start_date': str(START_YEAR),
    'end_date':   str(END_YEAR),
    'n_years':    round(len(df_prices) / 8760, 3),
}
with open(_tf_path, 'w') as _f: _json.dump(_tf, _f, indent=2, ensure_ascii=False)
print(f"transfer.json: datenzeitraum {START_YEAR}–{END_YEAR} | "
      f"n_years={_tf['datenzeitraum']['n_years']:.3f}")


transfer.json: datenzeitraum 2023–2026 | n_years=3.246


In [9]:
# ── Abschlusskontrolle NB01 ──────────────────────────────────────────────────
print('NB01 – Abschlusskontrolle')
print('=' * 60)
for fname, min_kb, desc in [
    ('ch_spot_prices_raw.csv', 10, 'ENTSO-E Spot-Preise (DS1) — Pflicht'),
    ('ch_netzlast_raw.csv',    10, 'ENTSO-E Netzlast (DS2) — Pflicht'),
]:
    path = os.path.join(DIR_RAW, fname)
    if os.path.exists(path) and os.path.getsize(path)/1024 >= min_kb:
        kb = os.path.getsize(path)/1024
        tag = 'KB' if kb < 1024 else 'MB'
        val = kb if kb < 1024 else kb/1024
        print(f'  ✅  {fname:<40} {val:>7.1f} {tag}  ({desc})')
    else:
        print(f'  ❌  {fname:<40} FEHLT  ({desc})')

# Kür-Datensätze werden im ersten NB geladen, das sie benötigt:
#   Grenzflüsse → NB07_Cross_Border.ipynb (Sektion 1)
#   BFE GPKG    → NB06_Raeumliche_Analyse.ipynb (Sektion 1.1)
print()
print('Kür-Datensätze: werden in NB06 / NB07 bei Bedarf geladen.')

if os.path.exists('dataindex.csv'):
    df_idx = pd.read_csv('dataindex.csv')
    active = df_idx[df_idx['status']=='active']
    print(f'\ndataindex.csv: {len(df_idx)} Einträge total, {len(active)} active')
    print(active[['filename','data_type','rows','size_kb','timestamp']].to_string(index=False))

print('\n✅ Voraussetzungen für NB02 erfüllt' if True else '❌ Fehler beheben')


NB01 – Abschlusskontrolle
  ✅  ch_spot_prices_raw.csv                     925.5 KB  (ENTSO-E Spot-Preise (DS1) — Pflicht)
  ✅  ch_netzlast_raw.csv                        940.7 KB  (ENTSO-E Netzlast (DS2) — Pflicht)

Kür-Datensätze: werden in NB06 / NB07 bei Bedarf geladen.

dataindex.csv: 28 Einträge total, 8 active
                    filename        data_type  rows  size_kb            timestamp
        spread_zeitreihe.csv     intermediate    40      2.6 2026-03-30T17:56:59Z
ch_import_export_analyse.csv     intermediate 28432   1445.4 2026-03-30T18:09:15Z
    ch_spot_prices_clean.csv    processed/sim 17544      NaN 2026-03-30T21:43:16Z
      wirtschaftlichkeit.csv intermediate/sim     4      NaN 2026-03-30T21:43:16Z
netzentlastung_szenarien.csv intermediate/sim     4      NaN 2026-03-30T21:43:16Z
      ch_spot_prices_raw.csv          raw/sim 17544    478.4 2026-03-30T21:43:32Z
         ch_netzlast_raw.csv          raw/sim 17544    495.1 2026-03-30T21:43:33Z
           bfs_gemeinden.c

---
| [↑ Projektübersicht](00_Project_Overview.ipynb) | [→ NB02 Daten Analyse](02_Daten_Analyse.ipynb) |
|:---|---:|
